# Assignment 3 – extensions: motor model, 3-state filter, IMU

**Mobile Robotics, KU Leuven** · Team: Cogilniceanu David (r0973221), Toghrul Alibayov (r0874286)

Offline only; the mandatory filter and the firmware (`Assignment3_LKF.ipynb`) are unchanged. Data: `M1_lifted` (voltage steps, robot lifted) and the floor drives W1–W3.

In [1]:
# Reuse data, calibration and filter of the main notebook: run its code cells without output
import nbformat, io, contextlib, warnings, matplotlib.pyplot as plt
_show = plt.show; plt.show = lambda *a, **k: None                        # no figures from the main notebook here
try:
    with contextlib.redirect_stdout(io.StringIO()), warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for _c in nbformat.read('Assignment3_LKF.ipynb', 4).cells:
            if _c.cell_type == 'code':
                exec(_c.source)
finally:
    plt.show = _show; plt.close('all')
from sklearn.linear_model import LinearRegression
from mr_common import imu_to_robot, PSI

## 1. Motor model

$V = L\dot I + R_aI + K_e n\omega$, $K_tI = J\dot\omega_m + c\,\omega_m$ ($n$ = 256/9). $L$ is neglected: $L/R_a$ = 23 µs ≪ 10 ms. The result is first order; per wheel, discretised: $\omega_{k+1} = \alpha\,\omega_k + \beta\,V_k$, $\tau = -\Delta t/\ln\alpha$, gain $\beta/(1-\alpha)$, fitted by linear regression.

**Datasheet comparison.** With $J = J_m$ and $c = 0$ the datasheet predicts $\tau$ = 11.3 ms and 2.43 rad/s/V; the lifted motor is slower (18 ms) and weaker (2.1 rad/s/V). With the datasheet $K_t, K_e, R_a$ this means $J ≈ 1.9\,J_m$ (gearbox, wheel) and a viscous friction of ≈ 15 % of the electrical damping; the electrical equation $V = R_aI + K_en\omega$ fitted on the logged current gives $K_e$ ≈ 0.016 V s/rad (+10 %) and $R_a$ ≈ 15–16 Ω (current scale and H-bridge included), and with those $J ≈ 1.1$–$1.2\,J_m$. The steady state also shows dry friction: $|\omega| ≈ 2.17(|V| - 0.18\text{ V})$. **On the floor** the robot's mass and rolling friction make the motor slower and weaker; with the A2 run R4 (6 V) the floor has the same slope as lifted but a larger offset (≈ 0.46 V).

In [2]:
KE_DS, KT_DS, RA_DS, L_DS, JM_DS, N_GEAR = 0.0145, 0.0145, 8.77, 203e-6, 2.7e-7, 256 / 9   # datasheet
M1 = load_log(DATA_A3 / 'M1_lifted.csv', firmware='identification_a3')

def fit_ab(w, V):
    # least squares for w[k+1] = alpha w[k] + beta V[k] (no intercept)
    X = np.column_stack([w[:-1], V[:-1]])
    reg = LinearRegression(fit_intercept=False).fit(X, w[1:])
    return reg.coef_, reg.score(X, w[1:])

def sim_ab(V, a, b, w0=0.0):
    # simulate the model from the voltages only
    w = np.zeros(len(V)); w[0] = w0
    for k in range(len(V) - 1):
        w[k + 1] = a * w[k] + b * V[k]
    return w

MOTOR = {}
for wh in 'AB':
    w, V = M1['w' + wh].values, M1['V' + wh].values
    (a, b), r2 = fit_ab(w, V)
    MOTOR[wh] = dict(alpha=a, beta=b, tau=-DT / np.log(a), gain=b / (1 - a), r2=r2, rmse_sim=np.sqrt(np.mean((sim_ab(V, a, b) - w)**2)))
lifted = pd.DataFrame(MOTOR).T
print('Lifted (M1):')
print(pd.DataFrame({'alpha': lifted.alpha.round(3), 'beta': lifted.beta.round(3), 'tau [ms]': (1e3 * lifted.tau).round(1),
                    'gain [rad/s/V]': lifted.gain.round(2), 'R2': lifted.r2.round(4), 'sim RMSE [rad/s]': lifted.rmse_sim.round(2)}).to_string())

FLOOR = []
for f in sorted(DATA_A3.glob('W*_start*_end*.csv')):
    df = load_log(f, firmware='identification_a3'); on = np.flatnonzero(df.VA.values != 0)
    FLOOR.append(dict(name=f.stem[:2], seg=df.iloc[on[0] - 20:on[-1] + 120].reset_index(drop=True)))
rows, MOTOR_FLOOR = [], {}
for wh in 'AB':
    st_f, st_m, rise_f, rise_m, rmse = [], [], [], [], []
    for run in FLOOR:
        w, V = run['seg']['w' + wh].values, run['seg']['V' + wh].values
        ws = sim_ab(V, MOTOR[wh]['alpha'], MOTOR[wh]['beta']); k_on = np.flatnonzero(V != 0)[0]
        steady = (V != 0) & (np.arange(len(V)) > k_on + 40)
        st_f.append(w[steady].mean()); st_m.append(ws[steady].mean()); rmse.append(np.sqrt(np.mean((ws - w)**2)))
        rise_f.append(np.argmax(w > 0.63 * w[steady].mean()) - k_on); rise_m.append(np.argmax(ws > 0.63 * ws[steady].mean()) - k_on)
    X = np.vstack([np.column_stack([r['seg']['w' + wh].values[:-1], r['seg']['V' + wh].values[:-1]]) for r in FLOOR])
    y = np.concatenate([r['seg']['w' + wh].values[1:] for r in FLOOR])
    a, b = LinearRegression(fit_intercept=False).fit(X, y).coef_
    MOTOR_FLOOR[wh] = dict(alpha=a, beta=b)
    rows.append({'wheel': wh, 'steady floor / lifted model [rad/s]': f'{np.mean(st_f):.2f} / {np.mean(st_m):.2f}',
                 '63 % rise floor / model [samples]': f'{np.mean(rise_f):.0f} / {np.mean(rise_m):.0f}',
                 'lifted model RMSE [rad/s]': np.mean(rmse), 'floor fit tau [ms]': round(-1e3 * DT / np.log(a), 1), 'floor fit gain [rad/s/V]': b / (1 - a)})
print('\nFloor (W1-W3, 3.48 V):')
print(pd.DataFrame(rows).round(2).to_string(index=False))

# second floor point: A2 run R4 at 6 V (another day) -> slope and dry-friction offset on the floor (wheel A)
r4 = pd.concat([window(R4, a, b) for a, b in [(31.9, 32.45), (33.0, 33.6)]])
wA_W = np.mean([r['seg'].wA[r['seg'].VA != 0].iloc[40:].mean() for r in FLOOR])
sl = (r4.wA.mean() - wA_W) / (r4.VA.mean() - 3.482)
print(f'floor, wheel A through (3.48 V, {wA_W:.2f} rad/s) and ({r4.VA.mean():.2f} V, {r4.wA.mean():.2f} rad/s): '
      f'slope {sl:.2f} rad/s/V, offset {3.482 - wA_W/sl:.2f} V')

Lifted (M1):
   alpha   beta  tau [ms]  gain [rad/s/V]      R2  sim RMSE [rad/s]
A  0.578  0.893      18.2            2.11  0.9997              0.16
B  0.574  0.894      18.0            2.10  0.9996              0.17

Floor (W1-W3, 3.48 V):
wheel steady floor / lifted model [rad/s] 63 % rise floor / model [samples]  lifted model RMSE [rad/s]  floor fit tau [ms]  floor fit gain [rad/s/V]
    A                         6.58 / 7.36                             5 / 2                       0.67                25.1                      1.87
    B                         6.55 / 7.30                             5 / 2                       0.63                24.2                      1.87
floor, wheel A through (3.48 V, 6.58 rad/s) and (5.98 V, 12.01 rad/s): slope 2.18 rad/s/V, offset 0.46 V


## 2. Three-state filter: $\mathbf x = [-d/R_\text{wheel},\ \omega_A,\ \omega_B]$

Dry friction is put into the input, $u' = V - V_0\,\mathrm{sign}(V)$ with $V_0$ = 0.46 V (floor), so the filter stays linear (as rotating the accelerations in A2). One $\alpha, \beta$ for both wheels, fitted on the floor with $u'$:

$$\mathbf F = \begin{bmatrix}1 & \tfrac{\Delta t}{2} & \tfrac{\Delta t}{2}\\ 0 & \alpha & 0\\ 0 & 0 & \alpha\end{bmatrix},\ \mathbf B = \begin{bmatrix}0 & 0\\ \beta & 0\\ 0 & \beta\end{bmatrix},\ \mathbf H = \begin{bmatrix}-R_\text{wheel} & 0 & 0\\ 0 & 1 & 0\\ 0 & 0 & 1\end{bmatrix}$$

$\mathbf Q$: $\omega$ rows from the variance of the one-step prediction residuals of the floor model, distance row = A2 value (slip shows up as model error in $\omega$). $\mathbf R$: calibrated distance as in the main filter (same gate), encoders from quantisation (0.351/√12 = 0.10 rad/s; the A2 value 0.18 rad/s alone would exceed the residual variance).

In [3]:
V0_FLOOR = 0.46                                                    # [V] dry friction on the floor (indicative)
uprime = lambda V: V - V0_FLOOR * np.sign(V)
SIG_ENC_Q = 2 * np.pi / 1792 / DT / np.sqrt(12)                    # encoder quantisation [rad/s]

def floor_fit(runs):
    # one alpha, beta for both wheels on the floor, input u'; returns alpha, beta, one-step residual variance
    X, y = [], []
    for r in runs:
        for wh in 'AB':
            w, V = r['seg']['w' + wh].values, r['seg']['V' + wh].values
            X.append(np.column_stack([w[:-1], uprime(V[:-1])])); y.append(w[1:])
    X, y = np.vstack(X), np.concatenate(y)
    reg = LinearRegression(fit_intercept=False).fit(X, y)
    return reg.coef_[0], reg.coef_[1], np.var(y - X @ reg.coef_)

A3S, B3S, QW3S = floor_fit(FLOOR)
print(f"alpha = {A3S:.3f}, beta = {B3S:.3f} (tau {1e3*-DT/np.log(A3S):.1f} ms, slope {B3S/(1-A3S):.2f} rad/s/V), one-step residual std {np.sqrt(QW3S):.2f} rad/s")

def kf3(m, VA, VB, wA, wB, x0, a, b, q_w, r_enc, drop_A=None):
    # correction (distance with the gate, encoders), then prediction with u'
    Fm = np.array([[1, DT / 2, DT / 2], [0, a, 0], [0, 0, a]]); Bm = np.array([[0, 0], [b, 0], [0, b]])
    Hm = np.array([[-R_WHEEL, 0, 0], [0, 1, 0], [0, 0, 1]]); Qm = np.diag([Q, q_w, q_w])
    n = len(m); x = np.array([x0, 0.0, 0.0]); P = np.diag([P0_STATE, r_enc, r_enc]); rejected = 0
    o = dict(x=np.zeros((n, 3)), nis3=np.full(n, np.nan), nis_ch=np.full((n, 3), np.nan), rej_3s=np.zeros(n, bool))
    for k in range(n):
        z = np.array([calibrate(m[k]), wA[k], wB[k]]); Rk = np.diag([R_of_m(m[k]), r_enc, r_enc])
        use = [True, drop_A is None or not drop_A[k], True]
        if not (SHARP_B < m[k] < M_RAW_MAX) or not (D_MIN <= z[0] <= D_MAX):
            use[0] = False
        elif (z[0] - Hm[0] @ x)**2 > GATE * (Hm[0] @ P @ Hm[0] + Rk[0, 0]):
            rejected += 1
            if rejected > MAX_REJECT:
                P[0, 0] = P_RESYNC; rejected = 0
            else:
                use[0] = False; o['rej_3s'][k] = True
        else:
            rejected = 0
        idx = [i for i in range(3) if use[i]]
        Hs = Hm[idx]; nu = z[idx] - Hs @ x; S = Hs @ P @ Hs.T + Rk[np.ix_(idx, idx)]
        if len(idx) == 3:
            o['nis3'][k] = nu @ np.linalg.solve(S, nu)
        for j, i in enumerate(idx):
            o['nis_ch'][k, i] = nu[j]**2 / S[j, j]
        K = P @ Hs.T @ np.linalg.inv(S); x = x + K @ nu; P = (np.eye(3) - K @ Hs) @ P
        o['x'][k] = x
        x = Fm @ x + Bm @ np.array([uprime(VA[k]), uprime(VB[k])]); P = Fm @ P @ Fm.T + Qm
    return o

alpha = 0.666, beta = 0.718 (tau 24.6 ms, slope 2.15 rad/s/V), one-step residual std 0.20 rad/s


## 3. IMU filter: $\mathbf x = [-d/R_\text{wheel},\ v]$, input = forward acceleration

$$\mathbf F = \begin{bmatrix}1 & \Delta t\\ 0 & 1\end{bmatrix},\ \mathbf B = \frac{1}{R_\text{wheel}}\begin{bmatrix}\Delta t^2/2\\ \Delta t\end{bmatrix},\ \mathbf Q = \mathbf B\sigma_a^2\mathbf B^\mathsf T,\ \mathbf H = \begin{bmatrix}-R_\text{wheel} & 0\\ 0 & 1\end{bmatrix}$$

$a_f$ from the A2 IMU calibration (forward axis at PSI ≈ 181° in IMU axes), bias from the first 5 s of each run; lever arm neglected (straight drives, yaw rate ≈ 0). Measurements: calibrated distance (same gate) and mean wheel speed ($R$ = $0.10/\sqrt2$ rad/s). $\sigma_a$: static (A2 R1) or driving (IMU minus wheel acceleration while driving).

In [4]:
R1s = load_log(DATA_A2 / 'R1_static.csv')
SIG_A_STATIC = imu_to_robot(R1s.ax.values, R1s.ay.values, PSI)[0].std()
SIG_V_ENC = SIG_ENC_Q / np.sqrt(2)

def imu_forward(df):
    st = window(df, 0.5, 5.5)                                              # standing still: bias of this run
    return imu_to_robot((df.ax - st.ax.mean()).values, (df.ay - st.ay.mean()).values, PSI)[0]

diff = []
for r in WRUNS:
    df = load_log(DATA_A3 / (r['name'] + '.csv'), firmware='identification_a3')
    v = R_WHEEL * ((df.wA + df.wB) / 2).rolling(5, center=True, min_periods=1).mean().values
    diff.append(imu_forward(df)[r['drive']] - np.gradient(v, df.t.values)[r['drive']])
SIG_A_DRIVE = np.concatenate(diff).std()
print(f'sigma_a static {SIG_A_STATIC:.3f} m/s^2, driving {SIG_A_DRIVE:.2f} m/s^2 ({SIG_A_DRIVE/SIG_A_STATIC:.0f} x)')

def kf_imu(m, a_f, w_mean, x0, sig_a, drop=None):
    Fm = np.array([[1, DT], [0, 1]]); Bm = np.array([DT**2 / 2, DT]) / R_WHEEL; Qm = np.outer(Bm, Bm) * sig_a**2
    Hm = np.array([[-R_WHEEL, 0], [0, 1]])
    n = len(m); x = np.array([x0, 0.0]); P = np.diag([P0_STATE, SIG_V_ENC**2]); rejected = 0
    o = dict(x=np.zeros((n, 2)), nis=np.full(n, np.nan), nis_ch=np.full((n, 2), np.nan), rej_3s=np.zeros(n, bool))
    for k in range(n):
        z = np.array([calibrate(m[k]), w_mean[k]]); Rk = np.diag([R_of_m(m[k]), SIG_V_ENC**2])
        use = [True, drop is None or not drop[k]]
        if not (SHARP_B < m[k] < M_RAW_MAX) or not (D_MIN <= z[0] <= D_MAX):
            use[0] = False
        elif (z[0] - Hm[0] @ x)**2 > GATE * (Hm[0] @ P @ Hm[0] + Rk[0, 0]):
            rejected += 1
            if rejected > MAX_REJECT:
                P[0, 0] = P_RESYNC; rejected = 0
            else:
                use[0] = False; o['rej_3s'][k] = True
        else:
            rejected = 0
        idx = [i for i in range(2) if use[i]]
        if idx:
            Hs = Hm[idx]; nu = z[idx] - Hs @ x; S = Hs @ P @ Hs.T + Rk[np.ix_(idx, idx)]
            if len(idx) == 2:
                o['nis'][k] = nu @ np.linalg.solve(S, nu)
            for j, i in enumerate(idx):
                o['nis_ch'][k, i] = nu[j]**2 / S[j, j]
            K = P @ Hs.T @ np.linalg.inv(S); x = x + K @ nu; P = (np.eye(2) - K @ Hs) @ P
        o['x'][k] = x
        x = Fm @ x + Bm * a_f[k]; P = Fm @ P @ Fm.T + Qm
    return o

sigma_a static 0.010 m/s^2, driving 0.73 m/s^2 (72 x)


In [5]:
# Comparison on W1-W3: calibration (and alpha, beta) held out per run; encoder A missing for 0.3 s mid-drive
res = {}
for j, r in enumerate(WRUNS):
    use_cal(fit_cal(cal_pts[~cal_pts.source.str.startswith(r['name'][:2])]))
    df = load_log(DATA_A3 / (r['name'] + '.csv'), firmware='identification_a3'); drv = r['drive']; af = imu_forward(df)
    wm = ((df.wA + df.wB) / 2).values; x0 = r['d_start'] / -R_WHEEL
    dr = np.arange(len(df))[drv]; gap = np.zeros(len(df), bool); gap[dr[40]:dr[70]] = True
    wA_bad = np.where(gap, 0.0, df.wA.values)                              # what a missing encoder reads
    a_j, b_j, q_j = floor_fit([FLOOR[i] for i in range(len(FLOOR)) if i != j])
    o1, met1, _ = w_metrics(r, Q_w)
    o1g = lkf(df.front.values, (wA_bad + df.wB.values) / 2, Q_w, x0, P0, gate=GATE)
    o3 = kf3(df.front.values, df.VA.values, df.VB.values, df.wA.values, df.wB.values, x0, a_j, b_j, q_j, SIG_ENC_Q**2)
    o3g = kf3(df.front.values, df.VA.values, df.VB.values, wA_bad, df.wB.values, x0, a_j, b_j, q_j, SIG_ENC_Q**2, drop_A=gap)
    runs = {'mandatory (1 state)': (H * o1['x'], met1['nis_drive'], 1, o1['rej_3s'][drv].sum(), H * o1g['x']),
            '3-state, motor model': (H * o3['x'][:, 0], np.nanmean(o3['nis3'][drv]), 3, o3['rej_3s'][drv].sum(), H * o3g['x'][:, 0])}
    for lab, sa in [('IMU, static sigma_a', SIG_A_STATIC), ('IMU, driving sigma_a', SIG_A_DRIVE)]:
        oi, oig = kf_imu(df.front.values, af, wm, x0, sa), kf_imu(df.front.values, af, wm, x0, sa, drop=gap)
        runs[lab] = (H * oi['x'][:, 0], np.nanmean(oi['nis'][drv]), 2, oi['rej_3s'][drv].sum(), H * oig['x'][:, 0])
    for lab, (d, nis, dim, rej, dg) in runs.items():
        e = res.setdefault(lab, dict(end=[], nis=[], dim=dim, rej=0, gap=[]))
        e['end'].append(1e3 * (d[-1] - r['d_end'])); e['nis'].append(nis); e['rej'] += int(rej); e['gap'].append(1e3 * np.max(np.abs(dg - d)))
use_cal(CAL_ALL)
print(pd.DataFrame([{'filter': lab, 'end error W1/W2/W3 [mm]': ' / '.join(f'{v:+.1f}' for v in e['end']),
                     'NIS driving (ideal)': f"{np.mean(e['nis']):.2f} ({e['dim']})", '3-sigma rej. driving': e['rej'],
                     'encoder A missing: max change [mm]': ' / '.join(f'{v:.1f}' for v in e['gap'])} for lab, e in res.items()]).to_string(index=False))

              filter end error W1/W2/W3 [mm] NIS driving (ideal)  3-sigma rej. driving encoder A missing: max change [mm]
 mandatory (1 state)      +3.0 / +0.6 / +5.6            0.92 (1)                    11                 30.3 / 31.4 / 27.2
3-state, motor model      +3.0 / +0.6 / +5.6            3.41 (3)                    16                    0.9 / 0.8 / 0.2
 IMU, static sigma_a      +0.6 / -1.0 / +2.6           84.40 (2)                    82                    1.4 / 1.0 / 1.4
IMU, driving sigma_a      +3.0 / +0.6 / +5.6            5.40 (2)                    74                    8.0 / 5.4 / 2.6


## 4. Conclusions

* **No gain in distance:** all filters end at the same error vs tape (set by the sensor calibration); the IMU filter is over-confident while driving (static $\sigma_a$: far too confident, its gate rejects the sensor; driving $\sigma_a$: still above its NIS dimension), for the A2 reasons: accelerometer bias drift and gravity leaking into the forward axis when the robot pitches.
* **Missing encoder:** the mandatory filter integrates the wrong speed and drifts by ≈ 30 mm; the motor model keeps the distance within 1 mm, the IMU within a few mm. The motor model is the better backup; detecting a wrong (not missing) encoder reading would need an innovation gate on the encoder channels.
* **Limitations:** $\alpha, \beta$ and especially $V_0$ come from one floor voltage (plus R4 at 6 V, another day); the first-order model misses the two-stage rise on the floor and the brief negative speed after $V = 0$.